In [1]:
# Standard Headers
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

In [2]:
# Read in the data
train_data = pd.read_csv("data/train.csv", skipinitialspace=True, low_memory=False, na_values = ["UNKNOWN", "Unknown", "UNK"])

# What percent of data values are null
print("Percent Null:", train_data.isnull().sum().sum() / (len(train_data) * len(train_data.columns)) * 100)

# Null amount for each column
print(train_data.isnull().sum())

Percent Null: 30.561646519429953
INDEX_NR                     0
INCIDENT_DATE                0
INCIDENT_MONTH               0
INCIDENT_YEAR                0
TIME                    132042
TIME_OF_DAY             133971
AIRPORT_ID                   0
AIRPORT                  40684
LATITUDE                 40744
LONGITUDE                40747
RUNWAY                   75586
STATE                    40744
FAAREGION                40744
LOCATION                267965
OPID                     86692
OPERATOR                 86692
REG                     118999
FLT                     161442
AIRCRAFT                 86942
AMA                      88896
AMO                     116813
EMA                     102421
EMO                     113176
AC_CLASS                 87315
AC_MASS                  87446
TYPE_ENG                 87786
NUM_ENGS                 87697
ENG_1_POS                87717
ENG_2_POS               102509
ENG_3_POS               294822
ENG_4_POS               303909
PHASE_

For each column, drop the column if it has 50% or more of its data missing

In [3]:
# For each column, drop the column if it has 50% or more of its data missing
print("Before:", len(train_data.columns))

for col in train_data:
    if train_data[col].isnull().sum() >= len(train_data) / 2:
        train_data.drop(col, inplace=True, axis=1)
        print(col)

print("After:", len(train_data.columns))

Before: 55
LOCATION
FLT
ENG_3_POS
ENG_4_POS
HEIGHT
SPEED
SKY
PRECIPITATION
BIRD_BAND_NUMBER
WARNED
NUM_SEEN
ENROUTE_STATE
After: 43


Drop highly correlated columns
- AIRPORT overlaps with AIRPORT_ID / coordinates
- OPERATOR overlaps with OPID
- SPECIES overlaps with SPECIES_ID

In [4]:
print("Before:", len(train_data.columns))
train_data.drop("AIRPORT", inplace=True, axis=1)
train_data.drop("OPERATOR", inplace=True, axis=1)
train_data.drop("SPECIES", inplace=True, axis=1)
print("After:", len(train_data.columns))

Before: 43
After: 40


Drop columns we do not need to use

In [5]:
print("Before:", len(train_data.columns))

DROP_COLS = [
    "REMARKS", "REMAINS_SENT", "COMMENTS", "SOURCE", 
    "PERSON", "LUPDATE", "TRANSFER", "NUM_STRUCK", 
    "FAAREGION", "INCIDENT_DATE", "REG", "INDEX_NR", 
    "ENG_1_POS", "ENG_2_POS", "AIRCRAFT", "AMA", 
    "AMO", "EMA", "EMO", "AC_CLASS"
]
for col in DROP_COLS:
    train_data.drop(col, inplace=True, axis=1)

print("After:", len(train_data.columns))

Before: 40
After: 20


Reasoning for dropping the columns above

In [6]:
"""
REMARKS, COMMENTS: 
Additional notes on the incident that would be difficult to quantify within a ML model

REMAINS_SENT, SOURCE, PERSON, LUPDATE, TRANSFER, REG:
Not very predictive of if there was damage to the aircraft

NUM_STRUCK:
89% of the data was the same, and almost all of the rest was corrupted

FAAREGION:
We decided to use LONG and LAT instead

INCIDENT_DATE: 
This is less relevant than other time values that we decided to keep

INDEX_NR: 
This is a separate index and would skew our result if kept

ENG_1_POS, ENG_2_POS, AIRCRAFT, AMA, AMO, EMA, EMO:
Dropped due to a combination of many missing values and a difficulty to generalize this across different aircrafts

AC_CLASS: 
70% of the data is an aircraft, 2 records total are helicopters, rest is unknown. So this is not very helpful
"""

'\nREMARKS, COMMENTS: \nAdditional notes on the incident that would be difficult to quantify within a ML model\n\nREMAINS_SENT, SOURCE, PERSON, LUPDATE, TRANSFER, REG:\nNot very predictive of if there was damage to the aircraft\n\nNUM_STRUCK:\n89% of the data was the same, and almost all of the rest was corrupted\n\nFAAREGION:\nWe decided to use LONG and LAT instead\n\nINCIDENT_DATE: \nThis is less relevant than other time values that we decided to keep\n\nINDEX_NR: \nThis is a separate index and would skew our result if kept\n\nENG_1_POS, ENG_2_POS, AIRCRAFT, AMA, AMO, EMA, EMO:\nDropped due to a combination of many missing values and a difficulty to generalize this across different aircrafts\n\nAC_CLASS: \n70% of the data is an aircraft, 2 records total are helicopters, rest is unknown. So this is not very helpful\n'

Dealing with remaining null values.

In [7]:
print(train_data.isnull().sum()[train_data.isnull().sum() > 0] / len(train_data) * 100)

TIME               42.985500
TIME_OF_DAY        43.613475
LATITUDE           13.263971
LONGITUDE          13.264947
RUNWAY             24.606580
STATE              13.263971
OPID               28.222073
AC_MASS            28.467533
TYPE_ENG           28.578218
NUM_ENGS           28.549245
PHASE_OF_FLIGHT    39.381075
DISTANCE           33.629687
SIZE               10.908008
dtype: float64


Filling missing numeric values with median of that column

In [8]:
print("Before:", len(train_data.columns))

# Find the comma errors where latitude and longitude were combined
comma_errors = train_data["LATITUDE"].str.contains(",", na=False)
vals_with_com_errs = train_data.loc[comma_errors, "LATITUDE"].str.split(",", expand=True)

# Fix the comma errors
train_data.loc[comma_errors, "LATITUDE"] = vals_with_com_errs[0]
train_data.loc[comma_errors, "LONGITUDE"] = vals_with_com_errs[1]

# Make LAT and LONG numeric floats
train_data["LATITUDE"] = pd.to_numeric(train_data["LATITUDE"], errors="coerce")
train_data["LONGITUDE"] = pd.to_numeric(train_data["LONGITUDE"], errors="coerce")

# Fill in the missing values for LAT and LONG, drop AIRPORT_ID and STATE
for col in ["LATITUDE", "LONGITUDE"]:
    train_data[col] = train_data[col].fillna(
        train_data.groupby("AIRPORT_ID")[col].transform("median")
    )

    train_data[col] = train_data[col].fillna(
        train_data.groupby("STATE")[col].transform("median")
    )

    train_data[col] = train_data[col].fillna(train_data[col].median())
train_data.drop("AIRPORT_ID", inplace=True, axis=1)
train_data.drop("STATE", inplace=True, axis=1)

# Extracting out the hour value from the time column
train_data["TIME"] = pd.to_datetime(train_data["TIME"], format="%H:%M", errors="coerce").dt.round("h").dt.hour

# Filling missing TIME values with the median value of it's TIME_OF_DAY
train_data["TIME"] = train_data.groupby("TIME_OF_DAY")["TIME"].transform(
    lambda x: x.fillna(x.median())
).round().astype("Int64")


# Filling rest of missing TIME values with median time
train_data["TIME"] = train_data["TIME"].fillna(train_data["TIME"].median())


# Drop TIME_OF_DAY column  
train_data.drop("TIME_OF_DAY", inplace=True, axis=1)

# Use PHASE_OF_FLIGHT to fill in nans for DISTANCE
train_data["DISTANCE"] = train_data["DISTANCE"].fillna(
    train_data.groupby("PHASE_OF_FLIGHT")["DISTANCE"].transform("median")
)

# Just use median DISTANCE for nans if PHASE_OF_FLIGHT is also missing for this data point
train_data["DISTANCE"] = train_data["DISTANCE"].fillna(train_data["DISTANCE"].median())

print("After:", len(train_data.columns))
print(train_data.isnull().sum() / len(train_data) * 100)

Before: 20
After: 17
INCIDENT_MONTH           0.000000
INCIDENT_YEAR            0.000000
TIME                     0.000000
LATITUDE                 0.000000
LONGITUDE                0.000000
RUNWAY                  24.606580
OPID                    28.222073
AC_MASS                 28.467533
TYPE_ENG                28.578218
NUM_ENGS                28.549245
PHASE_OF_FLIGHT         39.381075
DISTANCE                 0.000000
SPECIES_ID               0.000000
OUT_OF_RANGE_SPECIES     0.000000
REMAINS_COLLECTED        0.000000
SIZE                    10.908008
INDICATED_DAMAGE         0.000000
dtype: float64


Filling remaining missing values

In [9]:
print("Before:", len(train_data.columns))

# Convert runway to is or isn't airborne
train_data["AIRBORNE"] = train_data["RUNWAY"].isnull().astype(int)
train_data.drop("RUNWAY", inplace=True, axis=1)


# Fill in missing SIZE, fill all nans since less than .2% come from non "UNKB" SPECIES_ID
train_data["SIZE"] = train_data["SIZE"].fillna("UNK")
train_data.drop("SPECIES_ID", inplace=True, axis=1)

# Recude cardinality of OPID and convert missing values
top_ops = train_data["OPID"].value_counts().nlargest(10).index
train_data["OPID"] = train_data["OPID"].where(
    train_data["OPID"].isin(top_ops), "UNK"
)

# Add missing values for PHASE_OF_FLIGHT
train_data["PHASE_OF_FLIGHT"] = train_data["PHASE_OF_FLIGHT"].fillna("MISSING")

# Fill in missing values for NUM_ENGS
train_data["NUM_ENGS"] = train_data["NUM_ENGS"].fillna(train_data["NUM_ENGS"].median())

# Fill in missing values for AC_MASS
train_data["AC_MASS"] = train_data["AC_MASS"].fillna(train_data["AC_MASS"].median())

# Fill in missing values for TYPE_ENGS, use mode since categorical
train_data["TYPE_ENG"] = train_data["TYPE_ENG"].fillna(train_data["TYPE_ENG"].mode()[0])

print("After:", len(train_data.columns))
print(train_data.isnull().sum() / len(train_data) * 100)

Before: 17
After: 16
INCIDENT_MONTH          0.0
INCIDENT_YEAR           0.0
TIME                    0.0
LATITUDE                0.0
LONGITUDE               0.0
OPID                    0.0
AC_MASS                 0.0
TYPE_ENG                0.0
NUM_ENGS                0.0
PHASE_OF_FLIGHT         0.0
DISTANCE                0.0
OUT_OF_RANGE_SPECIES    0.0
REMAINS_COLLECTED       0.0
SIZE                    0.0
INDICATED_DAMAGE        0.0
AIRBORNE                0.0
dtype: float64


## Encoding categorical features
1. First, we identify which features are categorical vs numerical.

In [19]:
categorical_cols = []
numeric_cols = []

for col in train_data.columns:
    # Try converting column to numeric
    converted = pd.to_numeric(train_data[col], errors='coerce')
    
    # If any portion becomes NaN, it's categorical
    num_na = converted.isnull().sum()
    
    if num_na > 1:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)

print("Categorical Columns:")
print(categorical_cols)

print("\nNumeric Columns:")
print(numeric_cols)

print("\nTypes of the column data:")
print(train_data.dtypes)

Categorical Columns:
['OPID', 'TYPE_ENG', 'PHASE_OF_FLIGHT', 'SIZE']

Numeric Columns:
['INCIDENT_MONTH', 'INCIDENT_YEAR', 'TIME', 'LATITUDE', 'LONGITUDE', 'AC_MASS', 'NUM_ENGS', 'DISTANCE', 'OUT_OF_RANGE_SPECIES', 'REMAINS_COLLECTED', 'INDICATED_DAMAGE', 'AIRBORNE']

Types of the column data:
INCIDENT_MONTH            int64
INCIDENT_YEAR             int64
TIME                      Int64
LATITUDE                float64
LONGITUDE               float64
OPID                     object
AC_MASS                 float64
TYPE_ENG                 object
NUM_ENGS                float64
PHASE_OF_FLIGHT          object
DISTANCE                float64
OUT_OF_RANGE_SPECIES      int64
REMAINS_COLLECTED         int64
SIZE                     object
INDICATED_DAMAGE          int64
AIRBORNE                  int64
dtype: object


2. Next, we one-hot encode the features that are identified as categorical using get_dummies.

In [23]:
X = train_data.drop("INDICATED_DAMAGE", axis=1)
print("Number of rows and cols before encoding:", X.shape)

y = train_data["INDICATED_DAMAGE"]

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=False)

print("Number of rows and cols after encoding:", X_encoded.shape)

Number of rows and cols before encoding: (307178, 15)
Number of rows and cols after encoding: (307178, 45)


3. We noticed that by doing one-hot encoding, the number of columns tripled. So we decided to try doing a combination of one-hot encoding and frequency encoding to reduce this number.

In [26]:
# We want to identify high-cardinality columns (many unique values) 
# to use frequency encoding instead

high_card_cols = []
low_card_cols = []

for col in categorical_cols:
    num_unique = train_data[col].nunique()
    
    # Threshold: > 10 unique values, high cardinality
    if num_unique > 10:
        high_card_cols.append(col)
    else:
        low_card_cols.append(col)

print("High-cardinality columns (frequency encode):", high_card_cols)
print("Low-cardinality columns (one-hot encode):", low_card_cols)


# Frequency encode for high-cardinality columns
for col in high_card_cols:
    freq = train_data[col].value_counts(normalize=True)
    X[col] = X[col].map(freq)


# One-hot encode for low-cardinality columns
X_encoded = pd.get_dummies(X, columns=low_card_cols, drop_first=False)
print("Final shape after encoding:", X_encoded.shape)

High-cardinality columns (frequency encode): ['OPID', 'PHASE_OF_FLIGHT']
Low-cardinality columns (one-hot encode): ['TYPE_ENG', 'SIZE']
Final shape after encoding: (307178, 24)
